# Ingest circuits.csv file
### 1. Read the file using spark dataframe reader API
### 2. Add Metadata Columns 
-       Source File
-       Ingestion Timestamp
### 3. Write to bronze delta table

### Step 1 - Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType,
)

circuits_schema = StructType(
    [
        StructField("circuitId", StringType()),
        StructField("url", StringType()),
        StructField("circuitName", StringType()),
        StructField("lat", DoubleType()),
        StructField("long", DoubleType()),
        StructField("locality", StringType()),
        StructField("country", StringType()),
    ]
)

In [0]:
circuits_df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(circuits_schema)
    .load("/Volumes/formula1/landing/files/circuits.csv")
)

display(circuits_df)

In [0]:
from pyspark.sql import functions as F

circuits_final_df = (
    circuits_df
        .withColumn("Ingestion_Timestamp", F.current_timestamp())
        .withColumn("Source_File", F.col('_metadata.file_path'))
)

In [0]:
display(circuits_final_df)

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode("overwrite")
    .saveAsTable('formula1.bronze.circuits')
)

In [0]:
%sql
select * from formula1.bronze.circuits